# PetriDish V2-B — MAMMAL DTI predictions for HerbCheck

Inlines IBM's preprocessing logic (from `mammal/examples/dti_bindingdb_kd/task.py`) directly — avoids importing `pl_data_module.py` which transitively requires `tdc`/`tiledbsoma` (un-installable on Windows).

In [1]:
import torch, sys
print(f'Python   : {sys.version.split()[0]}')
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device   : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

Python   : 3.11.9
PyTorch  : 2.6.0+cu124
CUDA     : True
Device   : NVIDIA GeForce RTX 3060 Laptop GPU
VRAM     : 6.4 GB


In [2]:
from fuse.data.tokenizers.modular_tokenizer.op import ModularTokenizerOp
from mammal.model import Mammal
from mammal.keys import (
    ENCODER_INPUTS_STR, ENCODER_INPUTS_TOKENS, ENCODER_INPUTS_ATTENTION_MASK,
    ENCODER_INPUTS_SCALARS, SCALARS_PREDICTION_HEAD_LOGITS,
)

MODEL_ID = 'ibm/biomed.omics.bl.sm.ma-ted-458m.dti_bindingdb_pkd'
NORM_Y_MEAN = 5.79384684128215
NORM_Y_STD  = 1.33808027428196

print(f'Loading {MODEL_ID} ...')
tokenizer_op = ModularTokenizerOp.from_pretrained(MODEL_ID)
nn_model = Mammal.from_pretrained(MODEL_ID)
nn_model.eval().to(DEVICE)
print(f'Loaded. Parameters: {sum(p.numel() for p in nn_model.parameters())/1e6:.0f}M')

Loading ibm/biomed.omics.bl.sm.ma-ted-458m.dti_bindingdb_pkd ...

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

gene_tokenizer.json: 0.00B [00:00, ?B/s]

(…)th_aug_4272372_samples_balanced_1_1.json: 0.00B [00:00, ?B/s]

C:\MLProject\bioreason\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Shailesh\.cache\huggingface\hub\models--ibm--biomed.omics.bl.sm.ma-ted-458m.dti_bindingdb_pkd. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


...

model.safetensors:   0%|          | 0.00/1.83G [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

Attempting to load model from dir: pretrained_model_name_or_path='C:\\Users\\Shailesh\\.cache\\huggingface\\hub\\models--ibm--biomed.omics.bl.sm.ma-ted-458m.dti_bindingdb_pkd\\snapshots\\67164e29cdfb1a5cb8b8479b817f11b73c899ba7'

Loaded. Parameters: 458M

In [3]:
import requests

CYP_UNIPROT = {
    'CYP1A2':  'P05177',  'CYP2B6':  'P20813',  'CYP2C8':  'P10632',  'CYP2C9':  'P11712',
    'CYP2C19': 'P33261',  'CYP2D6':  'P10635',  'CYP2E1':  'P05181',  'CYP3A4':  'P08684',
}

def fetch_seq(acc):
    r = requests.get(f'https://rest.uniprot.org/uniprotkb/{acc}.fasta', timeout=15)
    r.raise_for_status()
    return ''.join(r.text.strip().split('\n')[1:])

CYP_SEQUENCES = {name: fetch_seq(acc) for name, acc in CYP_UNIPROT.items()}
for n, s in CYP_SEQUENCES.items():
    print(f'{n:8s} ({CYP_UNIPROT[n]}) : {len(s)} aa')

CYP1A2   (P05177) : 516 aa

CYP2B6   (P20813) : 491 aa

CYP2C8   (P10632) : 490 aa

CYP2C9   (P11712) : 490 aa

CYP2C19  (P33261) : 490 aa

CYP2D6   (P10635) : 497 aa

CYP2E1   (P05181) : 493 aa

CYP3A4   (P08684) : 503 aa

In [4]:
import pandas as pd
from pathlib import Path

ROOT = Path('..').resolve()
df = pd.read_csv(ROOT / 'api' / 'data' / 'imppat_smiles.csv')
compounds = df[df['smiles'].notna() & (df['smiles'].str.strip() != '')].copy()
print(f'{len(compounds)} of {len(df)} compounds have SMILES.')

24 of 30 compounds have SMILES.

In [5]:
# Inlined from mammal/examples/dti_bindingdb_kd/task.py — bypasses tdc import
TARGET_MAX_SEQ_LEN = 1250
DRUG_MAX_SEQ_LEN = 256
ENCODER_INPUT_MAX_SEQ_LEN = 1512

def build_prompt(target_seq, drug_seq):
    return (
        '<@TOKENIZER-TYPE=AA><MASK>'
        f'<@TOKENIZER-TYPE=AA@MAX-LEN={TARGET_MAX_SEQ_LEN}><MOLECULAR_ENTITY><MOLECULAR_ENTITY_GENERAL_PROTEIN><SEQUENCE_NATURAL_START>{target_seq}<SEQUENCE_NATURAL_END>'
        f'<@TOKENIZER-TYPE=SMILES@MAX-LEN={DRUG_MAX_SEQ_LEN}><MOLECULAR_ENTITY><MOLECULAR_ENTITY_SMALL_MOLECULE><SEQUENCE_NATURAL_START>{drug_seq}<SEQUENCE_NATURAL_END>'
        '<EOS>'
    )

def predict_pkd(drug_smiles, protein_seq):
    sample = {ENCODER_INPUTS_STR: build_prompt(protein_seq, drug_smiles)}
    tokenizer_op(
        sample,
        key_in=ENCODER_INPUTS_STR,
        key_out_tokens_ids=ENCODER_INPUTS_TOKENS,
        key_out_attention_mask=ENCODER_INPUTS_ATTENTION_MASK,
        max_seq_len=ENCODER_INPUT_MAX_SEQ_LEN,
        key_out_scalars=ENCODER_INPUTS_SCALARS,
    )
    sample[ENCODER_INPUTS_TOKENS] = torch.tensor(sample[ENCODER_INPUTS_TOKENS], device=nn_model.device)
    sample[ENCODER_INPUTS_ATTENTION_MASK] = torch.tensor(sample[ENCODER_INPUTS_ATTENTION_MASK], device=nn_model.device)
    with torch.no_grad():
        batch = nn_model.forward_encoder_only([sample])
    scalars_preds = batch[SCALARS_PREDICTION_HEAD_LOGITS]
    pkd = float((scalars_preds[:, 0] * NORM_Y_STD + NORM_Y_MEAN)[0])
    return pkd

# Sanity check
test_smiles = compounds[compounds['compound_name'] == 'Curcumin']['smiles'].iloc[0]
print(f'Curcumin × CYP3A4: pKd = {predict_pkd(test_smiles, CYP_SEQUENCES["CYP3A4"]):.3f}')
print('(Expected: 4.5 to 7.5 — Curcumin is a known CYP3A4 inhibitor)')

Curcumin × CYP3A4: pKd = 5.587

(Expected: 4.5 to 7.5 — Curcumin is a known CYP3A4 inhibitor)

In [6]:
from tqdm.auto import tqdm
import time

def classify_pkd(pkd):
    if pkd >= 8.0: return 'strong'
    if pkd >= 6.0: return 'moderate'
    if pkd >= 4.0: return 'weak'
    return 'non-binder'

results, errors = [], []
t0 = time.time()
for _, row in tqdm(compounds.iterrows(), total=len(compounds), desc='Compounds'):
    for cyp_name, cyp_seq in CYP_SEQUENCES.items():
        try:
            pkd = predict_pkd(row.smiles, cyp_seq)
        except Exception as e:
            errors.append((row.compound_name, cyp_name, str(e)[:120]))
            continue
        results.append({
            'imppat_id': row.imppat_id,
            'compound_name': row.compound_name,
            'cyp': cyp_name,
            'predicted_pkd': round(pkd, 3),
            'predicted_ic50_nM': round(10 ** (9 - pkd), 1) if pkd > 0 else None,
            'binding_class': classify_pkd(pkd),
            'binding_likely': pkd >= 5.0,
            'model': 'MAMMAL 458M DTI BindingDB-pKd',
            'computed_at': pd.Timestamp.utcnow().isoformat(),
        })
elapsed = time.time() - t0
print(f'\n{len(results)} predictions in {elapsed:.1f}s. Errors: {len(errors)}')
for e in errors[:5]:
    print(f'  {e[0]} × {e[1]}: {e[2]}')
preds = pd.DataFrame(results)
preds.head()

Compounds:   0%|          | 0/24 [00:00<?, ?it/s]


192 predictions in 37.4s. Errors: 0

,imppat_id,compound_name,cyp,predicted_pkd,predicted_ic50_nM,binding_class,binding_likely,model,computed_at
0,IMPPAT001,Curcumin,CYP1A2,5.639,2298.4,weak,True,MAMMAL 458M DTI BindingDB-pKd,2026-05-14T16:15:32.362827+00:00
1,IMPPAT001,Curcumin,CYP2B6,5.459,3473.1,weak,True,MAMMAL 458M DTI BindingDB-pKd,2026-05-14T16:15:32.555871+00:00
2,IMPPAT001,Curcumin,CYP2C8,5.792,1614.1,weak,True,MAMMAL 458M DTI BindingDB-pKd,2026-05-14T16:15:32.747320+00:00
3,IMPPAT001,Curcumin,CYP2C9,5.690,2042.4,weak,True,MAMMAL 458M DTI BindingDB-pKd,2026-05-14T16:15:32.938872+00:00
4,IMPPAT001,Curcumin,CYP2C19,5.714,1931.8,weak,True,MAMMAL 458M DTI BindingDB-pKd,2026-05-14T16:15:33.137943+00:00


In [7]:
out_path = ROOT / 'api' / 'data' / 'mammal_predictions.csv'
preds.to_csv(out_path, index=False)
print(f'Saved {len(preds)} rows to {out_path}\n')
print(preds['binding_class'].value_counts())
print(f'\nMean pKd: {preds.predicted_pkd.mean():.2f}')
print('\nTop 10 strongest binders:')
print(preds.nlargest(10, 'predicted_pkd')[['compound_name', 'cyp', 'predicted_pkd', 'binding_class']].to_string(index=False))

Saved 192 rows to C:\MLProject\bioreason\api\data\mammal_predictions.csv


binding_class
weak    192
Name: count, dtype: int64


Mean pKd: 5.66


Top 10 strongest binders:

  compound_name     cyp  predicted_pkd binding_class
 Boswellic acid  CYP2C8          5.920          weak
Andrographolide  CYP2C8          5.909          weak
           EGCG  CYP2C8          5.885          weak
   Withaferin A  CYP2C8          5.868          weak
    Galantamine  CYP2C8          5.868          weak
 Boswellic acid  CYP2E1          5.855          weak
     Mangiferin  CYP2C8          5.846          weak
      Forskolin  CYP2C8          5.846          weak
   Ellagic acid  CYP2C8          5.840          weak
 Boswellic acid CYP2C19          5.836          weak